#### open anacoonda prompt and start the server of 'surprise' .. if not present create 'surprise' environment and launch jupyter in that 

In [1]:
import surprise
import pandas as pd
import numpy as np
import os
os.chdir("E:/Machine_Learning/Cases_Rec_Sys/filmtrust/")

In [2]:
ratings = pd.read_csv('ratings.txt',sep=" ",names=['uid','iid','rating'])
ratings.head()

,uid,iid,rating
0,1,1,2.0
1,1,2,4.0
2,1,3,3.5
3,1,4,3.0
4,1,5,4.0


In [5]:
lowest_rating=ratings['rating'].min()
highest_rating=ratings['rating'].max()
lowest_rating,highest_rating

(np.float64(0.5), np.float64(4.0))

In [7]:
reader = surprise.Reader(rating_scale = (lowest_rating,highest_rating))
data = surprise.Dataset.load_from_df(ratings,reader)
type(data)

surprise.dataset.DatasetAutoFolds

In [15]:
similarity_options = {'name' :'cosine','user_based': True}
algo = surprise.KNNBasic(sim_options = similarity_options)  #Default k = 40
output = algo.fit(data.build_full_trainset())
#Calculates the expected ratings for all the users

Computing the cosine similarity matrix...
Done computing similarity matrix.


In [10]:
from surprise.model_selection import train_test_split

In [13]:
trainset , testset = train_test_split(data,test_size = 0.25,random_state=26)
similarity_options = {'name' :'cosine','user_based': True}
algo = surprise.KNNBasic(sim_options = similarity_options,k=30) 
algo.fit(trainset)
predictions = algo.test(testset)
surprise.accuracy.rmse(predictions)

Computing the cosine similarity matrix...
Done computing similarity matrix.
RMSE: 0.9334


np.float64(0.9333665668939722)

### Grid Search CV

In [16]:
from surprise.model_selection import GridSearchCV
from surprise.model_selection import KFold

In [17]:
# param_grid = { 'k': [25,40,50,65,75,85],'user_based': [True]}  ->for user based 
# param_grid = { 'k': [25,40,50,65,75,85],'user_based': [False]}  ->for item based
param_grid = { 'k': [25,40,50,65,75,85],'user_based': [False,True]}
kFold = KFold(n_splits= 5,random_state=26,shuffle=True)
gs = GridSearchCV(surprise.KNNBasic,param_grid,measures=['rmse','mae'],cv=kFold)
gs.fit(data)

Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computi

In [19]:
print(gs.best_score['rmse'])
print(gs.best_params['rmse'])

0.8642083344321752
{'k': 65, 'user_based': False}


In [20]:
print(gs.best_score['mae'])
print(gs.best_params['mae'])

0.6555882949237102
{'k': 50, 'user_based': False}


### Building beast model on whole data

In [21]:
algo = gs.best_estimator['rmse']
algo.fit(data.build_full_trainset())

Computing the msd similarity matrix...
Done computing similarity matrix.


### Total number of items

In [24]:
iids = ratings['iid'].unique()
print(len(iids))

2071


#### Consider any user

In [22]:
a = [2,4,5,7,8,9]
b = [4,7,9]
np.setdiff1d(a,b)

array([2, 5, 8])

In [28]:
user = 50
u_iid = ratings[ratings['uid'] ==user]['iid'].unique()
print("list of items rated by user : \n",u_iid)
print("No. of items rated by user {0} : {1} ".format(user,len(u_iid)))
iids_to_predict = np.setdiff1d(iids,u_iid)
print("items not rated by the user or those items which the expected ratings are to be predicted : \n")
print(iids_to_predict)

list of items rated by user : 
 [  8 211   3   2 219 234  12 254 250 207  11 253 236  84  10   7 233  13
   1   5   6 252 241 216 257 206   4 217   9 215 213  17 255 220 121 245
 239 251 235]
No. of items rated by user 50 : 39 
items not rated by the user or those items which the expected ratings are to be predicted : 

[  14   15   16 ... 2069 2070 2071]


In [33]:
testset = [[user,iid,0.] for iid in iids_to_predict]
predictions = algo.test(testset)
exp_ratings = [(predictions[i].iid,predictions[i].est) for i in range(0,len(predictions))]
df_exp_ratings = pd.DataFrame(exp_ratings,columns=['iid','est_rating'])
df_exp_ratings.sort_values(by='est_rating',ascending=False).head(30)

,iid,est_rating
1096,1136,4.0
1978,2018,4.0
1974,2014,4.0
1973,2013,4.0
20,35,4.0
2024,2064,4.0
1119,1159,4.0
1914,1954,4.0
1580,1620,4.0
53,68,4.0


## Using KNNWithZScore instead of KNNBasic

In [34]:
param_grid = { 'k': [25,40,50,65,75,85],'user_based': [False,True]}
kFold = KFold(n_splits= 5,random_state=26,shuffle=True)
gs = GridSearchCV(surprise.KNNWithZScore,param_grid,measures=['rmse','mae'],cv=kFold)
gs.fit(data)

Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computi

In [35]:
print(gs.best_score['rmse'])
print(gs.best_params['rmse'])

0.82858464566139
{'k': 85, 'user_based': False}


In [36]:
algo = gs.best_estimator['rmse']
algo.fit(data.build_full_trainset())

Computing the msd similarity matrix...
Done computing similarity matrix.


### Total number of items

In [37]:
iids = ratings['iid'].unique()
print(len(iids))

2071


### Consider any user

In [38]:
a = [2,4,5,7,8,9]
b = [4,7,9]
np.setdiff1d(a,b)

array([2, 5, 8])

In [39]:
user = 50
u_iid = ratings[ratings['uid'] ==user]['iid'].unique()
print("list of items rated by user : \n",u_iid)
print("No. of items rated by user {0} : {1} ".format(user,len(u_iid)))
iids_to_predict = np.setdiff1d(iids,u_iid)
print("items not rated by the user or those items which the expected ratings are to be predicted : \n")
print(iids_to_predict)

list of items rated by user : 
 [  8 211   3   2 219 234  12 254 250 207  11 253 236  84  10   7 233  13
   1   5   6 252 241 216 257 206   4 217   9 215 213  17 255 220 121 245
 239 251 235]
No. of items rated by user 50 : 39 
items not rated by the user or those items which the expected ratings are to be predicted : 

[  14   15   16 ... 2069 2070 2071]


In [40]:
testset = [[user,iid,0.] for iid in iids_to_predict]
predictions = algo.test(testset)
exp_ratings = [(predictions[i].iid,predictions[i].est) for i in range(0,len(predictions))]
df_exp_ratings = pd.DataFrame(exp_ratings,columns=['iid','est_rating'])
df_exp_ratings.sort_values(by='est_rating',ascending=False).head(30)

,iid,est_rating
12,27,4.0
1119,1159,4.0
1932,1972,4.0
1933,1973,4.0
1149,1189,4.0
1323,1363,4.0
424,464,4.0
439,479,4.0
1259,1299,4.0
1544,1584,4.0
